In [ ]:
from __future__ import annotations

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import sys
import json
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------------------------------------------------------
# Jupyter / matplotlib setup
# -----------------------------------------------------------------------------
plt.style.use("default")
%matplotlib inline

# -----------------------------------------------------------------------------
# Resolve project root and imports
# -----------------------------------------------------------------------------
cwd = Path.cwd().resolve()
project_root = cwd
while not (project_root / "experiments" / "metric_loader.py").exists() and project_root != project_root.parent:
    project_root = project_root.parent

if not (project_root / "experiments" / "metric_loader.py").exists():
    raise RuntimeError(f"Could not locate EviTrack project root from {cwd}")

for path in (project_root, project_root / "experiments"):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from experiments.metric_loader import MetricLoader  # noqa: E402

# =============================================================================
# USER CONFIG
# =============================================================================
DATA_PATH = project_root / "results" / "MAIN_RUN_04_16_2026" / "doublewell_analytical"

# =============================================================================
# METRIC SELECTION — CHANGE THIS TO SWITCH METRICS
# =============================================================================
METRIC_TO_ANALYZE = "pll"  # OPTIONS: "pll", "mse", "ba"

# Metric configurations
METRIC_CONFIGS = {
    "pll": {
        "key": "pll",
        "name": "Predictive Log-Likelihood",
        "ylabel": "PLL",
        "clip": None,      # e.g. (-150.0, 5.0)
        "ylim": None,      # e.g. (-150.0, 5.0)
    },
    "mse": {
        "key": "obs_mse",
        "name": "Observation MSE",
        "ylabel": "MSE",
        "clip": None,      # e.g. (0.0, 8.0)
        "ylim": None,      # e.g. (0.0, 8.0)
    },
    "ba": {
        "key": "ba",
        "name": "Branch Accuracy",
        "ylabel": "Branch Accuracy",
        "clip": None,
        "ylim": (0.0, 1.0),  # BA is always 0-1
    },
}

# Validate selection
if METRIC_TO_ANALYZE not in METRIC_CONFIGS:
    raise ValueError(
        f"Invalid METRIC_TO_ANALYZE='{METRIC_TO_ANALYZE}'. "
        f"Must be one of: {list(METRIC_CONFIGS.keys())}"
    )

# Extract config for selected metric
METRIC_CFG = METRIC_CONFIGS[METRIC_TO_ANALYZE]
METRIC_KEY = METRIC_CFG["key"]
METRIC_NAME = METRIC_CFG["name"]
YLABEL = METRIC_CFG["ylabel"]
VALUE_CLIP = METRIC_CFG["clip"]
Y_LIMIT = METRIC_CFG["ylim"]

# =============================================================================
# ENGINE DISPLAY CONFIGURATION
# =============================================================================
# Centralized configuration for engine colors, aliases, and styles
# Edit "alias" to change display names in plots (e.g., "ET-J" instead of "EviTrack-J-Ginf")

ENGINE_DISPLAY_CONFIG = {
    "EviTrack-E-Ginf": {
        "color": "#1f77b4",  # blue
        "alias": "EviTrack-E-Ginf",
        "linestyle": "-",
        "linewidth": 2.8,
        "marker": None,
    },
    "EviTrack-J-Ginf": {
        "color": "#ff7f0e",  # orange
        "alias": "EviTrack-J-Ginf",
        "linestyle": "-",
        "linewidth": 2.8,
        "marker": None,
    },
    "EviTrack-E-MaxW": {
        "color": "#2ca02c",  # green
        "alias": "EviTrack-E-MaxW",
        "linestyle": "-",
        "linewidth": 2.2,
        "marker": None,
    },
    "EviTrack-J-MaxW": {
        "color": "#d62728",  # red
        "alias": "EviTrack-J-MaxW",
        "linestyle": "-",
        "linewidth": 2.2,
        "marker": None,
    },
    "SIS-PF": {
        "color": "#9467bd",  # purple
        "alias": "SIS-PF",
        "linestyle": "-",
        "linewidth": 2.2,
        "marker": None,
    },
    "Bootstrap-PF": {
        "color": "#8c564b",  # brown
        "alias": "Bootstrap-PF",
        "linestyle": "-",
        "linewidth": 2.2,
        "marker": None,
    },
}

# Engine ordering for tables and plots
ENGINE_ORDER_PREFERRED = [
    "EviTrack-E-Ginf",
    "EviTrack-J-Ginf",
    "EviTrack-E-MaxW",
    "EviTrack-J-MaxW",
    "SIS-PF",
    "Bootstrap-PF",
]

# =============================================================================
# PLOT CONFIGURATION
# =============================================================================

# Per-seed plot controls
PER_SEED_PLOT_CONFIG = {
    "show_title": True,           # Show suptitle
    "show_xlabel": True,          # Show x-axis labels
    "show_ylabel": True,          # Show y-axis label (leftmost subplot only)
    "show_legend": True,          # Show legend
    "figsize_per_bin": 5,         # Width per subplot (total width = n_bins * this)
    "fig_height": 4,              # Figure height
    "legend_ncol": 3,             # Number of legend columns
    "legend_loc": "upper center", # Legend location
    "legend_bbox": (0.5, -0.05),  # Legend bbox_to_anchor (negative y = below plot)
    "grid_alpha": 0.3,            # Grid transparency
    "dd_line_color": "black",     # Color of t=0 vertical line
    "dd_line_style": "--",        # Style of t=0 line
    "dd_line_width": 1.5,         # Width of t=0 line
    "band_alpha": 0.18,           # Transparency of uncertainty bands
}

# Aggregated plot controls
AGGREGATED_PLOT_CONFIG = {
    "show_title": True,
    "show_xlabel": True,
    "show_ylabel": True,
    "show_legend": True,
    "global_figsize": (10, 5),    # Size for "ALL" bin plot
    "figsize_per_bin": 5,         # Width per subplot for binned plots
    "fig_height": 4,              # Height for binned plots
    "legend_ncol": 3,
    "legend_loc": "upper center",
    "legend_bbox": (0.5, -0.05),
    "grid_alpha": 0.3,
    "dd_line_color": "black",
    "dd_line_style": "--",
    "dd_line_width": 1.5,
    "band_alpha": 0.18,
}

# =============================================================================
# ANALYSIS SETTINGS
# =============================================================================

# Leave as None / [] to use all available
ENGINE_NAMES   = None
SEED_IDX       = None   # e.g. 0 or [0,1,2] or None -> all
HORIZON_IDX    = None   # e.g. 2 or [0,1,2] or None -> all
DD_BIN_LABELS  = None   # e.g. ["30-80", "80-140"]  or None -> all

WINDOW = 20             # DD-aligned plot window: t_rel in [-WINDOW, +WINDOW]

# --- Per-seed analysis ---
SHOW_PER_SEED_TABLES = True    # Show tables for each seed individually
SHOW_PER_SEED_PLOTS  = True    # Show plots for each seed individually
PER_SEED_PLOT_REDUCE = "mean_std"  # "mean_std" or "median" for per-seed plots

# --- Seed-aggregated analysis ---
SHOW_AGGREGATED_SUMMARY_TABLE   = True  # Compact: "mean ± std" format
SHOW_AGGREGATED_BREAKDOWN_TABLE = True  # Expanded: individual seed values
SHOW_AGGREGATED_PLOTS           = True  # Cross-seed mean ± 95% CI
AGGREGATED_PLOT_REDUCE          = "mean_std"  # "mean_std" or "median" for aggregated plots

# =============================================================================
# HELPERS
# =============================================================================
def ensure_list_or_none(x):
    """Normalise scalar / sequence / empty -> list or None."""
    if x is None:
        return None
    if isinstance(x, (list, tuple, set, np.ndarray)):
        x = list(x)
        return None if len(x) == 0 else x
    return [x]


def sort_bin_labels(bin_labels: Sequence[str]) -> List[str]:
    return sorted(bin_labels, key=lambda x: int(str(x).split("-")[0]))


def preferred_engine_order(available_engines: Sequence[str], preferred: Sequence[str]) -> List[str]:
    available_engines = list(available_engines)
    ordered = [e for e in preferred if e in available_engines]
    ordered += [e for e in available_engines if e not in ordered]
    return ordered


def get_engine_style(engine: str) -> Dict[str, Any]:
    """Get plotting style for an engine."""
    if engine in ENGINE_DISPLAY_CONFIG:
        cfg = ENGINE_DISPLAY_CONFIG[engine]
        return {
            "color": cfg.get("color"),
            "linestyle": cfg.get("linestyle", "-"),
            "linewidth": cfg.get("linewidth", 2.0),
            "marker": cfg.get("marker"),
        }
    return {"linewidth": 2.0}


def get_engine_alias(engine: str) -> str:
    """Get display alias for an engine."""
    if engine in ENGINE_DISPLAY_CONFIG:
        return ENGINE_DISPLAY_CONFIG[engine].get("alias", engine)
    return engine


def load_dd_bin_ranges(dataset_dir: Path) -> Dict[str, tuple]:
    """Load DD time bin ranges from metadata.json."""
    metadata_path = dataset_dir / "metadata.json"
    if not metadata_path.exists():
        raise FileNotFoundError(f"Could not find metadata.json at {metadata_path}")

    with open(metadata_path, "r") as f:
        metadata = json.load(f)

    bin_start_indices = metadata["bin_start_indices"]

    bin_ranges = {}
    for bin_label in bin_start_indices.keys():
        parts = str(bin_label).split("-")
        if len(parts) != 2:
            raise ValueError(f"Could not parse DD bin label '{bin_label}'")
        min_time = int(parts[0])
        max_time = int(parts[1])
        bin_ranges[str(bin_label)] = (min_time, max_time)

    return bin_ranges


def stratify_by_dd_time(
    data_by_traj_seed: Dict,
    engines: Sequence[str],
    traj_indices: Sequence[int],
    seed_indices: Sequence[int],
    bin_ranges: Dict[str, tuple],
) -> Dict[str, List[int]]:
    """
    Stratify trajectories by true DD time using one reference engine/seed.
    Assumes dd_time_truth is trajectory-level truth and therefore invariant across engines.
    """
    if len(engines) == 0:
        raise ValueError("No engines available for DD stratification.")
    if len(seed_indices) == 0:
        raise ValueError("No seed indices available for DD stratification.")

    ref_engine = engines[0]
    ref_seed   = seed_indices[0]

    bin_to_trajs = {bin_label: [] for bin_label in bin_ranges.keys()}

    for traj_idx in traj_indices:
        dd_time = data_by_traj_seed[ref_engine][traj_idx][ref_seed]["dd_time_truth"]

        for bin_label, (min_time, max_time) in bin_ranges.items():
            if min_time <= dd_time < max_time:
                bin_to_trajs[bin_label].append(traj_idx)
                break
        # trajectories outside all bins are silently skipped

    return bin_to_trajs


def select_available_indices(requested, available, name: str):
    requested = ensure_list_or_none(requested)
    available  = list(available)
    if requested is None:
        return available

    missing = [x for x in requested if x not in available]
    if missing:
        raise ValueError(
            f"Requested {name} {missing} not available. Available {name}: {available}"
        )
    return requested


def filter_bin_dict(bin_to_trajs: Dict[str, List[int]], dd_bin_labels=None) -> Dict[str, List[int]]:
    dd_bin_labels = ensure_list_or_none(dd_bin_labels)
    if dd_bin_labels is None:
        return {k: list(v) for k, v in bin_to_trajs.items()}

    out = {}
    for label in dd_bin_labels:
        if label not in bin_to_trajs:
            raise ValueError(
                f"Requested DD bin '{label}' not found. Available bins: {list(bin_to_trajs.keys())}"
            )
        out[label] = list(bin_to_trajs[label])
    return out


def safe_stats_1d(x: np.ndarray) -> Dict[str, float]:
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    if x.size == 0:
        return {"mean": np.nan, "std": np.nan, "median": np.nan,
                "q25": np.nan, "q75": np.nan, "n": 0}
    return {
        "mean":   float(np.mean(x)),
        "std":    float(np.std(x, ddof=1)) if x.size > 1 else 0.0,
        "median": float(np.median(x)),
        "q25":    float(np.percentile(x, 25)),
        "q75":    float(np.percentile(x, 75)),
        "n":      int(x.size),
    }


def aggregate_seed_scalar_stats(values_by_seed: Dict[int, np.ndarray]) -> Dict[str, Any]:
    """
    Aggregate scalar-per-trajectory arrays across seeds.
    Returns both summary stats and individual seed values.
    """
    seed_means = []
    seed_indices = []

    for seed, arr in sorted(values_by_seed.items()):
        arr = np.asarray(arr, dtype=float)
        arr = arr[~np.isnan(arr)]
        if arr.size == 0:
            continue
        seed_means.append(arr.mean())
        seed_indices.append(seed)

    if len(seed_means) == 0:
        return {
            "mean_over_seeds": np.nan,
            "std_over_seeds": np.nan,
            "median_over_seeds": np.nan,
            "q25_over_seeds": np.nan,
            "q75_over_seeds": np.nan,
            "n_seeds": 0,
            "seed_values": {},
        }

    seed_means = np.asarray(seed_means, dtype=float)

    return {
        "mean_over_seeds":   float(np.mean(seed_means)),
        "std_over_seeds":    float(np.std(seed_means, ddof=1)) if seed_means.size > 1 else 0.0,
        "median_over_seeds": float(np.median(seed_means)),
        "q25_over_seeds":    float(np.percentile(seed_means, 25)),
        "q75_over_seeds":    float(np.percentile(seed_means, 75)),
        "n_seeds":           int(seed_means.size),
        "seed_values":       {seed_indices[i]: seed_means[i] for i in range(len(seed_means))},
    }


def pre_post_values_for_metric(
    data_by_traj_seed: Dict,
    engines: Sequence[str],
    traj_indices: Sequence[int],
    seed_indices: Sequence[int],
    metric_key: str,
) -> Dict[str, Dict[int, Dict[str, np.ndarray]]]:
    """
    Returns:
        out[engine][seed]["pre"]  -> 1-D array, one mean-per-trajectory (pre-DD)
        out[engine][seed]["post"] -> 1-D array, one mean-per-trajectory (post-DD)
    """
    out: Dict[str, Dict[int, Dict[str, np.ndarray]]] = {}

    for engine in engines:
        out[engine] = {}
        for seed in seed_indices:
            pre_vals, post_vals = [], []
            for traj_idx in traj_indices:
                metrics = data_by_traj_seed[engine][traj_idx][seed]
                dd_time = int(metrics["dd_time_truth"])
                arr     = np.asarray(metrics[metric_key], dtype=float)

                if dd_time < 0 or dd_time > len(arr):
                    continue

                pre_arr  = arr[:dd_time]
                post_arr = arr[dd_time:]
                pre_vals.append(float(np.nanmean(pre_arr))  if pre_arr.size  > 0 else np.nan)
                post_vals.append(float(np.nanmean(post_arr)) if post_arr.size > 0 else np.nan)

            out[engine][seed] = {
                "pre":  np.asarray(pre_vals,  dtype=float),
                "post": np.asarray(post_vals, dtype=float),
            }

    return out


def build_summary_tables_for_metric(
    data_by_traj_seed: Dict,
    engines: Sequence[str],
    seed_indices: Sequence[int],
    bin_to_trajs: Dict[str, List[int]],
    metric_key: str,
    show_per_seed_tables: bool = False,
    show_aggregated_summary: bool = True,
    show_aggregated_breakdown: bool = True,
) -> Dict[str, Any]:
    """
    Returns:
        per_seed_dfs: {bin_label: {seed: df}}
        aggregated_summary_dfs: {bin_label: df}  (compact: "mean ± std")
        aggregated_breakdown_dfs: {bin_label: df}  (expanded: individual seed values)
    """
    results: Dict[str, Any] = {
        "per_seed_dfs": {},
        "aggregated_summary_dfs": {},
        "aggregated_breakdown_dfs": {},
    }

    all_trajs = sorted({t for trajs in bin_to_trajs.values() for t in trajs})
    bins_with_all = {"ALL": all_trajs}
    bins_with_all.update(bin_to_trajs)

    for bin_label, traj_indices in bins_with_all.items():
        vals = pre_post_values_for_metric(
            data_by_traj_seed=data_by_traj_seed,
            engines=engines,
            traj_indices=traj_indices,
            seed_indices=seed_indices,
            metric_key=metric_key,
        )

        # --- Per-seed tables ---
        if show_per_seed_tables:
            results["per_seed_dfs"][bin_label] = {}
            for seed in seed_indices:
                rows = []
                for engine in engines:
                    pre_s  = safe_stats_1d(vals[engine][seed]["pre"])
                    post_s = safe_stats_1d(vals[engine][seed]["post"])
                    rows.append({
                        "engine":       engine,
                        "pre_mean":     pre_s["mean"],
                        "pre_std":      pre_s["std"],
                        "pre_median":   pre_s["median"],
                        "pre_q25":      pre_s["q25"],
                        "pre_q75":      pre_s["q75"],
                        "post_mean":    post_s["mean"],
                        "post_std":     post_s["std"],
                        "post_median":  post_s["median"],
                        "post_q25":     post_s["q25"],
                        "post_q75":     post_s["q75"],
                        "n_traj":       post_s["n"],
                    })
                results["per_seed_dfs"][bin_label][seed] = pd.DataFrame(rows)

        # --- Aggregated summary table (compact) ---
        if show_aggregated_summary:
            rows = []
            for engine in engines:
                pre_agg  = aggregate_seed_scalar_stats(
                    {s: vals[engine][s]["pre"]  for s in seed_indices})
                post_agg = aggregate_seed_scalar_stats(
                    {s: vals[engine][s]["post"] for s in seed_indices})

                pre_str = f"{pre_agg['mean_over_seeds']:.3f} ± {pre_agg['std_over_seeds']:.3f}"
                post_str = f"{post_agg['mean_over_seeds']:.3f} ± {post_agg['std_over_seeds']:.3f}"

                rows.append({
                    "engine":  engine,
                    "pre_DD":  pre_str,
                    "post_DD": post_str,
                    "n_seeds": post_agg["n_seeds"],
                })
            results["aggregated_summary_dfs"][bin_label] = pd.DataFrame(rows)

        # --- Aggregated breakdown table (expanded) ---
        if show_aggregated_breakdown:
            rows = []
            for engine in engines:
                pre_agg  = aggregate_seed_scalar_stats(
                    {s: vals[engine][s]["pre"]  for s in seed_indices})
                post_agg = aggregate_seed_scalar_stats(
                    {s: vals[engine][s]["post"] for s in seed_indices})

                row = {
                    "engine":       engine,
                    "post_mean":    post_agg["mean_over_seeds"],
                    "post_std":     post_agg["std_over_seeds"],
                    "post_median":  post_agg["median_over_seeds"],
                    "post_q25":     post_agg["q25_over_seeds"],
                    "post_q75":     post_agg["q75_over_seeds"],
                }

                # Add individual seed columns dynamically
                for seed in seed_indices:
                    row[f"seed_{seed}"] = post_agg["seed_values"].get(seed, np.nan)

                row["n_seeds"] = post_agg["n_seeds"]
                rows.append(row)

            results["aggregated_breakdown_dfs"][bin_label] = pd.DataFrame(rows)

    return results


def compute_dd_aligned_curves_per_seed(
    data_by_traj_seed: Dict,
    engines: Sequence[str],
    seed_indices: Sequence[int],
    bin_to_trajs: Dict[str, List[int]],
    metric_key: str,
    window: int = 20,
    reduce_traj: str = "mean_std",
) -> Dict[str, Dict[str, Dict[int, Dict[str, np.ndarray]]]]:
    """
    Returns:
        curves[bin_label][engine][seed] = {
            "center": [2*window+1],
            "lower":  [2*window+1],
            "upper":  [2*window+1],
            "n_traj": int,
            "t_rel":  [2*window+1],
        }
    Trajectories whose aligned window would go out of bounds are skipped.
    """
    curves: Dict[str, Dict[str, Dict[int, Dict[str, np.ndarray]]]] = {}

    for bin_label, traj_indices in bin_to_trajs.items():
        curves[bin_label] = {}

        for engine in engines:
            curves[bin_label][engine] = {}

            for seed in seed_indices:
                aligned = []
                for traj_idx in traj_indices:
                    metrics = data_by_traj_seed[engine][traj_idx][seed]
                    dd_time = int(metrics["dd_time_truth"])
                    arr     = np.asarray(metrics[metric_key], dtype=float)

                    start = dd_time - window
                    stop  = dd_time + window + 1

                    if start < 0 or stop > len(arr):
                        continue  # skip out-of-bounds

                    aligned.append(arr[start:stop])

                if len(aligned) == 0:
                    continue

                aligned = np.stack(aligned, axis=0)  # [N_traj, T_rel]

                n_valid = np.sum(~np.isnan(aligned), axis=0).astype(float)
                if reduce_traj == "mean_std":
                    center = np.nanmean(aligned, axis=0)
                    std    = np.where(n_valid > 1, np.nanstd(aligned, axis=0, ddof=1), 0.0)
                    lower  = center - std
                    upper  = center + std
                elif reduce_traj == "median":
                    center = np.nanmedian(aligned, axis=0)
                    lower  = np.nanpercentile(aligned, 25, axis=0)
                    upper  = np.nanpercentile(aligned, 75, axis=0)
                else:
                    raise ValueError(f"Unknown reduce_traj={reduce_traj}")

                curves[bin_label][engine][seed] = {
                    "center": center,
                    "lower":  lower,
                    "upper":  upper,
                    "n_traj": aligned.shape[0],
                    "t_rel":  np.arange(-window, window + 1),
                }

    return curves


def aggregate_curves_across_seeds(
    per_seed_curves: Dict[str, Dict[str, Dict[int, Dict[str, np.ndarray]]]],
    engines: Sequence[str],
    seed_indices: Sequence[int],
    reduce_mode: str = "mean_std",
) -> Dict[str, Dict[str, Dict[str, np.ndarray]]]:
    """
    Aggregate reduced per-seed curves across seeds.

    reduce_mode:
        "mean_std": center = mean, bands = mean ± std across seed curves
        "median": center = median, bands = Q25–Q75 across seed curves
    """
    agg: Dict[str, Dict[str, Dict[str, np.ndarray]]] = {}

    for bin_label in per_seed_curves.keys():
        agg[bin_label] = {}

        for engine in engines:
            seed_entries = [
                per_seed_curves[bin_label][engine][s]
                for s in seed_indices
                if engine in per_seed_curves.get(bin_label, {})
                   and s in per_seed_curves[bin_label][engine]
            ]

            if len(seed_entries) == 0:
                continue

            centers = np.stack([s["center"] for s in seed_entries], axis=0)
            t_rel   = seed_entries[0]["t_rel"]

            if reduce_mode == "mean_std":
                center = centers.mean(axis=0)
                std    = centers.std(axis=0, ddof=1) if centers.shape[0] > 1 else np.zeros_like(center)
                lower  = center - std
                upper  = center + std
            elif reduce_mode == "median":
                center = np.median(centers, axis=0)
                lower  = np.percentile(centers, 25, axis=0)
                upper  = np.percentile(centers, 75, axis=0)
            else:
                raise ValueError(f"Unknown reduce_mode={reduce_mode}")

            agg[bin_label][engine] = {
                "center":  center,
                "lower":   lower,
                "upper":   upper,
                "t_rel":   t_rel,
                "n_seeds": centers.shape[0],
            }

    return agg


def apply_clip(y, lo=None, hi=None):
    if lo is None and hi is None:
        return y
    lo = -np.inf if lo is None else lo
    hi =  np.inf if hi is None else hi
    return np.clip(y, lo, hi)


def display_df(
    df: pd.DataFrame,
    title: str = "",
    engine_order: Optional[Sequence[str]] = None,
    digits: int = 4,
):
    from IPython.display import display as ipy_display
    if df is None or len(df) == 0:
        print(f"{title}\n[empty]\n")
        return

    df_show = df.copy()
    if engine_order is not None and "engine" in df_show.columns:
        df_show["__order__"] = pd.Categorical(
            df_show["engine"], categories=list(engine_order), ordered=True
        )
        df_show = df_show.sort_values(["__order__", "engine"]).drop(columns="__order__")

    # Round only numeric columns (skip formatted strings like "mean ± std")
    numeric_cols = df_show.select_dtypes(include=[np.number]).columns
    df_show[numeric_cols] = df_show[numeric_cols].round(digits)

    if title:
        print(f"\n{'='*100}")
        print(title)
        print(f"{'='*100}")
    ipy_display(df_show.reset_index(drop=True))


def plot_per_seed_metric(
    per_seed_curves: Dict[str, Dict[str, Dict[int, Dict[str, np.ndarray]]]],
    engines: Sequence[str],
    seed_indices: Sequence[int],
    bin_order: Sequence[str],
    metric_name: str,
    ylabel: str,
    clip: Optional[tuple] = None,
    ylim: Optional[tuple] = None,
):
    """
    Generate one figure per seed, with bins as subplots.
    Each subplot shows all engines for that seed in that bin.
    """
    cfg = PER_SEED_PLOT_CONFIG

    for seed in seed_indices:
        # Only keep bins that have data for this seed and at least one engine
        active_bins = [
            b for b in bin_order
            if b in per_seed_curves
            and any(engine in per_seed_curves[b] and seed in per_seed_curves[b][engine]
                    for engine in engines)
        ]

        n_bins = len(active_bins)
        if n_bins == 0:
            print(f"[Seed {seed}] No bin data available for {metric_name}.")
            continue

        print(f"\nPer-seed plot (Seed {seed}): {metric_name} per DD bin, aggregated over trajectories within seed\n")

        fig, axes = plt.subplots(
            1, n_bins,
            figsize=(cfg["figsize_per_bin"] * n_bins, cfg["fig_height"]),
            sharey=True
        )
        if n_bins == 1:
            axes = [axes]

        for ax, bin_label in zip(axes, active_bins):
            for engine in engines:
                if (engine not in per_seed_curves[bin_label] or
                    seed not in per_seed_curves[bin_label][engine]):
                    continue

                s  = per_seed_curves[bin_label][engine][seed]
                x  = s["t_rel"]
                y  = apply_clip(s["center"].copy(), *(clip if clip else (None, None)))
                lo = apply_clip(s["lower"].copy(),  *(clip if clip else (None, None)))
                hi = apply_clip(s["upper"].copy(),  *(clip if clip else (None, None)))

                style = get_engine_style(engine)
                alias = get_engine_alias(engine)
                ax.plot(x, y, label=alias, **style)
                ax.fill_between(x, lo, hi, alpha=cfg["band_alpha"])

            ax.axvline(
                0,
                color=cfg["dd_line_color"],
                linestyle=cfg["dd_line_style"],
                linewidth=cfg["dd_line_width"],
                alpha=0.7
            )
            ax.set_title(f"{bin_label}")

            if cfg["show_xlabel"]:
                ax.set_xlabel("Time relative to true DD")

            ax.grid(True, alpha=cfg["grid_alpha"])

            if ylim is not None:
                ax.set_ylim(*ylim)

        if cfg["show_ylabel"]:
            axes[0].set_ylabel(ylabel)

        if cfg["show_legend"]:
            handles, labels = axes[0].get_legend_handles_labels()
            if handles:
                fig.legend(
                    handles, labels,
                    loc=cfg["legend_loc"],
                    bbox_to_anchor=cfg["legend_bbox"],
                    ncol=min(len(labels), cfg["legend_ncol"]),
                    frameon=True
                )

        if cfg["show_title"]:
            fig.suptitle(f"{metric_name} — Seed {seed}", fontsize=14)

        plt.tight_layout()
        plt.show()


def plot_aggregated_metric(
    agg_curves: Dict[str, Dict[str, Dict[str, np.ndarray]]],
    engines: Sequence[str],
    bin_order: Sequence[str],
    metric_name: str,
    ylabel: str,
    clip: Optional[tuple] = None,
    ylim: Optional[tuple] = None,
    show_global: bool = True,
    show_bins: bool = True,
):
    """
    Plot seed-aggregated curves.
    - If show_global: plot "ALL" bin separately
    - If show_bins: plot per-bin subplots
    """
    cfg = AGGREGATED_PLOT_CONFIG

    if show_global and "ALL" in agg_curves:
        print(f"\nGlobal plot: {metric_name} aggregated across all trajectories and seeds\n")

        fig, ax = plt.subplots(figsize=cfg["global_figsize"])

        for engine in engines:
            if engine not in agg_curves["ALL"]:
                continue

            s  = agg_curves["ALL"][engine]
            x  = s["t_rel"]
            y  = apply_clip(s["center"].copy(), *(clip if clip else (None, None)))
            lo = apply_clip(s["lower"].copy(),  *(clip if clip else (None, None)))
            hi = apply_clip(s["upper"].copy(),  *(clip if clip else (None, None)))

            style = get_engine_style(engine)
            alias = get_engine_alias(engine)
            ax.plot(x, y, label=alias, **style)
            ax.fill_between(x, lo, hi, alpha=cfg["band_alpha"])

        ax.axvline(
            0,
            color=cfg["dd_line_color"],
            linestyle=cfg["dd_line_style"],
            linewidth=cfg["dd_line_width"],
            alpha=0.7
        )

        if cfg["show_xlabel"]:
            ax.set_xlabel("Time relative to true DD")

        if cfg["show_ylabel"]:
            ax.set_ylabel(ylabel)

        if cfg["show_title"]:
            ax.set_title(f"{metric_name} aligned to true DD (all trajectories)")

        ax.grid(True, alpha=cfg["grid_alpha"])

        if ylim is not None:
            ax.set_ylim(*ylim)

        if cfg["show_legend"]:
            ax.legend(fontsize=9)

        plt.tight_layout()
        plt.show()

    if show_bins:
        # Only keep bins that have data for at least one engine
        active_bins = [b for b in bin_order if b in agg_curves and len(agg_curves[b]) > 0]
        n_bins = len(active_bins)
        if n_bins == 0:
            print(f"No bin data available for {metric_name}.")
            return

        print(f"\nBinned plots: {metric_name} per DD time bin, aggregated across seeds\n")

        fig, axes = plt.subplots(
            1, n_bins,
            figsize=(cfg["figsize_per_bin"] * n_bins, cfg["fig_height"]),
            sharey=True
        )
        if n_bins == 1:
            axes = [axes]

        for ax, bin_label in zip(axes, active_bins):
            for engine in engines:
                if engine not in agg_curves[bin_label]:
                    continue

                s  = agg_curves[bin_label][engine]
                x  = s["t_rel"]
                y  = apply_clip(s["center"].copy(), *(clip if clip else (None, None)))
                lo = apply_clip(s["lower"].copy(),  *(clip if clip else (None, None)))
                hi = apply_clip(s["upper"].copy(),  *(clip if clip else (None, None)))

                style = get_engine_style(engine)
                alias = get_engine_alias(engine)
                ax.plot(x, y, label=alias, **style)
                ax.fill_between(x, lo, hi, alpha=cfg["band_alpha"])

            ax.axvline(
                0,
                color=cfg["dd_line_color"],
                linestyle=cfg["dd_line_style"],
                linewidth=cfg["dd_line_width"],
                alpha=0.7
            )
            ax.set_title(f"DD bin: {bin_label}")

            if cfg["show_xlabel"]:
                ax.set_xlabel("Time relative to true DD")

            ax.grid(True, alpha=cfg["grid_alpha"])

            if ylim is not None:
                ax.set_ylim(*ylim)

        if cfg["show_ylabel"]:
            axes[0].set_ylabel(ylabel)

        if cfg["show_legend"]:
            handles, labels = axes[0].get_legend_handles_labels()
            if handles:
                fig.legend(
                    handles, labels,
                    loc=cfg["legend_loc"],
                    bbox_to_anchor=cfg["legend_bbox"],
                    ncol=min(len(labels), cfg["legend_ncol"]),
                    frameon=True
                )

        if cfg["show_title"]:
            fig.suptitle(f"{metric_name} aligned to true DD", fontsize=14)

        plt.tight_layout()
        plt.show()


# =============================================================================
# MAIN DRIVER
# =============================================================================
def run_forecasting_report(
    data_path: Path,
    metric_key: str,
    metric_name: str,
    ylabel: str,
    value_clip: Optional[tuple] = None,
    y_limit: Optional[tuple] = None,
    engine_names=None,
    seed_idx=None,
    horizon_idx=None,
    dd_bin_labels=None,
    window: int = 20,
    show_per_seed_tables: bool = False,
    show_per_seed_plots: bool = False,
    per_seed_plot_reduce: str = "mean_std",
    show_aggregated_summary: bool = True,
    show_aggregated_breakdown: bool = True,
    show_aggregated_plots: bool = True,
    aggregated_plot_reduce: str = "mean_std",
):
    results_dir = Path(data_path)
    replay_dir  = results_dir / "replay"
    dataset_dir = results_dir / "dataset"

    if not replay_dir.exists():
        raise FileNotFoundError(f"Replay directory not found: {replay_dir}")
    if not dataset_dir.exists():
        raise FileNotFoundError(f"Dataset directory not found: {dataset_dir}")

    # ------------------------------------------------------------------
    # 1) Load data
    # ------------------------------------------------------------------
    base_loader = MetricLoader(replay_dir=replay_dir, verbose=True)
    base_loader.load().organize(horizon_idx=0)

    available_engines  = list(base_loader.engines)
    available_seeds    = list(base_loader.all_seed_indices)
    available_horizons = list(range(len(base_loader.horizons)))

    selected_engines  = select_available_indices(engine_names, available_engines,  "engines")
    selected_seeds    = select_available_indices(seed_idx,     available_seeds,    "seed indices")
    selected_horizons = select_available_indices(horizon_idx,  available_horizons, "horizon indices")

    bin_ranges_all = load_dd_bin_ranges(dataset_dir)

    print(f"\n{'='*100}")
    print(f"FORECASTING REPORT: {metric_name.upper()}")
    print(f"{'='*100}")
    print(f"DATA_PATH           : {results_dir}")
    print(f"METRIC_KEY          : {metric_key}")
    print(f"SELECTED_ENGINES    : {selected_engines}")
    print(f"SELECTED_SEEDS      : {selected_seeds}")
    horizon_values = [int(base_loader.horizons[hi]) for hi in selected_horizons]
    print(f"SELECTED_HORIZONS   : {selected_horizons}  (values: {horizon_values})")
    print(f"REQUESTED_DD_BINS   : {'ALL' if dd_bin_labels is None else dd_bin_labels}")
    print(f"WINDOW              : {window}")
    print(f"PER_SEED_TABLES     : {show_per_seed_tables}")
    print(f"PER_SEED_PLOTS      : {show_per_seed_plots}")
    print(f"PER_SEED_REDUCE     : {per_seed_plot_reduce}")
    print(f"AGGREGATED_SUMMARY  : {show_aggregated_summary}")
    print(f"AGGREGATED_BREAKDOWN: {show_aggregated_breakdown}")
    print(f"AGGREGATED_PLOTS    : {show_aggregated_plots}")
    print(f"AGGREGATED_REDUCE   : {aggregated_plot_reduce}")
    print(f"{'='*100}\n")

    selected_engines = preferred_engine_order(selected_engines, ENGINE_ORDER_PREFERRED)

    # ------------------------------------------------------------------
    # 2) Loop over selected horizons
    # ------------------------------------------------------------------
    for h_idx in selected_horizons:
        H_value = int(base_loader.horizons[h_idx])

        print(f"\n{'#'*120}")
        print(f"HORIZON = {H_value}")
        print(f"{'#'*120}\n")

        loader = MetricLoader(replay_dir=replay_dir, verbose=False)
        loader.load().organize(horizon_idx=h_idx)

        data_by_traj_seed = loader.data_by_traj_seed

        engines_h = [e for e in selected_engines if e in loader.engines]
        seeds_h   = select_available_indices(selected_seeds, loader.all_seed_indices, "seed indices")
        trajs_h   = list(loader.all_traj_indices)

        bin_to_trajs_all = stratify_by_dd_time(
            data_by_traj_seed=data_by_traj_seed,
            engines=engines_h,
            traj_indices=trajs_h,
            seed_indices=seeds_h,
            bin_ranges=bin_ranges_all,
        )
        bin_to_trajs = filter_bin_dict(bin_to_trajs_all, dd_bin_labels=dd_bin_labels)
        bin_order    = sort_bin_labels(list(bin_to_trajs.keys()))

        print("DD bin counts:")
        for label in bin_order:
            print(f"  {label:>8}: {len(bin_to_trajs[label])}")

        # ------------------------------------------------------------------
        # TABLES
        # ------------------------------------------------------------------
        print(f"\n{'-'*120}")
        print(f"TABLES")
        print(f"{'-'*120}")

        tables = build_summary_tables_for_metric(
            data_by_traj_seed=data_by_traj_seed,
            engines=engines_h,
            seed_indices=seeds_h,
            bin_to_trajs=bin_to_trajs,
            metric_key=metric_key,
            show_per_seed_tables=show_per_seed_tables,
            show_aggregated_summary=show_aggregated_summary,
            show_aggregated_breakdown=show_aggregated_breakdown,
        )

        # Per-seed tables
        if show_per_seed_tables:
            for bin_label in ["ALL"] + bin_order:
                for seed in seeds_h:
                    df = tables["per_seed_dfs"].get(bin_label, {}).get(seed)
                    display_df(
                        df,
                        title=f"[H={H_value}] {metric_name} — {bin_label} (Seed {seed})",
                        engine_order=engines_h,
                    )

        # Aggregated summary tables (compact)
        if show_aggregated_summary:
            for bin_label in ["ALL"] + bin_order:
                df = tables["aggregated_summary_dfs"].get(bin_label)
                display_df(
                    df,
                    title=f"[H={H_value}] {metric_name} — {bin_label} (Summary: mean ± std)",
                    engine_order=engines_h,
                )

        # Aggregated breakdown tables (expanded)
        if show_aggregated_breakdown:
            for bin_label in ["ALL"] + bin_order:
                df = tables["aggregated_breakdown_dfs"].get(bin_label)
                display_df(
                    df,
                    title=f"[H={H_value}] {metric_name} — {bin_label} (Breakdown: individual seeds)",
                    engine_order=engines_h,
                )

        # ------------------------------------------------------------------
        # PLOTS
        # ------------------------------------------------------------------
        if show_per_seed_plots or show_aggregated_plots:
            # Include "ALL" pseudo-bin in bin_to_trajs
            all_selected_trajs = sorted({t for v in bin_to_trajs.values() for t in v})
            bin_to_trajs_with_all = {"ALL": all_selected_trajs}
            bin_to_trajs_with_all.update(bin_to_trajs)

            # Compute per-seed curves
            per_seed_curves = compute_dd_aligned_curves_per_seed(
                data_by_traj_seed=data_by_traj_seed,
                engines=engines_h,
                seed_indices=seeds_h,
                bin_to_trajs=bin_to_trajs_with_all,
                metric_key=metric_key,
                window=window,
                reduce_traj=per_seed_plot_reduce,
            )

            # Per-seed plots
            if show_per_seed_plots:
                print(f"\n{'-'*120}")
                print(f"PER-SEED PLOTS")
                print(f"{'-'*120}\n")

                plot_per_seed_metric(
                    per_seed_curves=per_seed_curves,
                    engines=engines_h,
                    seed_indices=seeds_h,
                    bin_order=["ALL"] + bin_order,
                    metric_name=f"[H={H_value}] {metric_name}",
                    ylabel=ylabel,
                    clip=value_clip,
                    ylim=y_limit,
                )

            # Aggregated plots
            if show_aggregated_plots:
                print(f"\n{'-'*120}")
                print(f"SEED-AGGREGATED PLOTS")
                print(f"{'-'*120}\n")

                agg_curves = aggregate_curves_across_seeds(
                    per_seed_curves=per_seed_curves,
                    engines=engines_h,
                    seed_indices=seeds_h,
                    reduce_mode=aggregated_plot_reduce,
                )

                plot_aggregated_metric(
                    agg_curves=agg_curves,
                    engines=engines_h,
                    bin_order=bin_order,
                    metric_name=f"[H={H_value}] {metric_name}",
                    ylabel=ylabel,
                    clip=value_clip,
                    ylim=y_limit,
                    show_global=True,
                    show_bins=True,
                )


# =============================================================================
# RUN
# =============================================================================
run_forecasting_report(
    data_path=DATA_PATH,
    metric_key=METRIC_KEY,
    metric_name=METRIC_NAME,
    ylabel=YLABEL,
    value_clip=VALUE_CLIP,
    y_limit=Y_LIMIT,
    engine_names=ENGINE_NAMES,
    seed_idx=SEED_IDX,
    horizon_idx=HORIZON_IDX,
    dd_bin_labels=DD_BIN_LABELS,
    window=WINDOW,
    show_per_seed_tables=SHOW_PER_SEED_TABLES,
    show_per_seed_plots=SHOW_PER_SEED_PLOTS,
    per_seed_plot_reduce=PER_SEED_PLOT_REDUCE,
    show_aggregated_summary=SHOW_AGGREGATED_SUMMARY_TABLE,
    show_aggregated_breakdown=SHOW_AGGREGATED_BREAKDOWN_TABLE,
    show_aggregated_plots=SHOW_AGGREGATED_PLOTS,
    aggregated_plot_reduce=AGGREGATED_PLOT_REDUCE,
)